# PyTorch Tensör Boyutu Manipülasyonu

Transformer mimarisinde çok sık karşılaşılan bir durum: `Linear` katmanı (matris çarpımı) her zaman 2 boyutlu girdi bekler, ama elimizdeki veri genelde `(batch, sequence, embedding)` şeklinde 3 boyutludur. Bu notebook'ta:

1. `(32, 10, 64)` boyutunda bir tensörü `(320, 64)` haline getirip,
2. `(64, 128)` boyutunda bir ağırlık matrisiyle çarpıp,
3. sonucu tekrar `(32, 10, 128)` haline döndürüyoruz.

Sonda `view` ile `reshape` arasındaki farkı ve `contiguous` kavramını açıklıyoruz.

In [1]:
import torch

torch.manual_seed(42)  # tekrarlanabilir sonuçlar için

## Adım 1 — Başlangıç tensörü: (batch=32, sequence=10, embedding=64)

Bunu, 32 cümlelik bir batch, her cümlede 10 token, her token'ın 64 boyutlu bir embedding vektörüyle temsil edildiği bir veri olarak düşünebiliriz.

In [2]:
x = torch.randn(32, 10, 64)
print(x.shape)

torch.Size([32, 10, 64])


## Adım 2 — (320, 64) boyutuna getir

`@` operatörü (matris çarpımı) 2 boyutlu matrisler bekler. `batch` ve `sequence` boyutlarını tek bir boyutta birleştiriyoruz: `32 * 10 = 320`. Yani artık "320 tane, her biri 64 boyutlu token vektörü var" diyoruz — hangi cümleye ait olduğu bilgisi geçici olarak kayboluyor, ama sayılar aynı kalıyor.

In [3]:
x_flat = x.view(32 * 10, 64)
print(x_flat.shape)

torch.Size([320, 64])


## Adım 3 — Ağırlık matrisi: (64, 128)

Bu, bir `nn.Linear(64, 128)` katmanının ağırlığına karşılık gelir: her 64 boyutlu token vektörünü 128 boyutlu bir temsile dönüştürür.

In [4]:
weight = torch.randn(64, 128)
print(weight.shape)

torch.Size([64, 128])


## Adım 4 — Matris çarpımı: (320, 64) @ (64, 128) = (320, 128)

Matris çarpımı kuralı: `(a, b) @ (b, c) = (a, c)`. Ortadaki boyutlar (`64`) birbirini "yiyor", elde `(320, 128)` kalıyor.

In [5]:
out_flat = x_flat @ weight
print(out_flat.shape)

torch.Size([320, 128])


## Adım 5 — Tekrar (32, 10, 128) haline getir

Çarpma işlemi bitti; şimdi "320"yi tekrar "32 cümle × 10 token" olarak ayırıyoruz. Sayılar değişmiyor, sadece nasıl gruplandığı (shape) değişiyor.

In [6]:
out = out_flat.view(32, 10, 128)
print(out.shape)

torch.Size([32, 10, 128])


## Neden hem `view` hem `reshape` var? `contiguous` nedir?

PyTorch'ta bir tensörün verisi, hafızada aslında **düz (1 boyutlu) bir sayı dizisi** olarak durur. `shape` (boyut) bilgisi, bu diziye nasıl "bakılacağını" söyleyen bir etikettir — verinin kendisi değil.

- **`view()`**: Veriyi hiç kopyalamadan, aynı hafıza bloğuna sadece farklı bir açıdan bakar (çok hızlıdır, ekstra bellek harcamaz). Ama bunun çalışabilmesi için verinin hafızada **contiguous** (satır-major sırada, aralıksız) olması gerekir. `transpose()`, `permute()` gibi işlemler veriyi fiziksel olarak taşımaz, sadece "okuma sırasını" değiştirir — bu da tensörü contiguous olmaktan çıkarır. Böyle bir tensörde `view()` çağırırsan `RuntimeError` alırsın.
- **`reshape()`**: Daha "toleranslı"dır — mümkünse `view()` gibi kopyasız çalışır, mümkün değilse (tensör contiguous değilse) otomatik olarak veriyi kopyalayıp yeniden düzenler.

Yani ikisi de "aynı işi" yapar ama:
- `view()` → daha hızlı/öngörülebilir, ama sadece contiguous tensörlerde çalışır, aksi halde hata verir.
- `reshape()` → her zaman çalışır, ama bazen (görünmeden) bir kopya oluşturarak ekstra bellek/performans maliyetine yol açabilir.

Pratik kural: Elindeki tensörün contiguous olduğundan eminsen (örneğin yeni oluşturulmuş, transpose/permute uygulanmamış bir tensörse) `view()` kullan; emin değilsen veya `transpose`/`permute` sonrası bir reshape yapıyorsan `reshape()` kullan.

### Önemli Not — contiguous olmayan bir tensörde `view()` neden patlar?

In [7]:
print('x_flat contiguous mi?', x_flat.is_contiguous())

x_transposed = x_flat.T  # transpose — veriyi taşımaz, sadece stride'ı değiştirir
print('transpose sonrası contiguous mi?', x_transposed.is_contiguous())

try:
    x_transposed.view(320, 64)  # x_flat ile aynı shape'e geri dönmeye çalışıyoruz
except RuntimeError as e:
    print('view() hatasi:', e)

# reshape() aynı durumda calisir (gerekirse kopyalar)
print('reshape() calisti, yeni shape:', x_transposed.reshape(320, 64).shape)

x_flat contiguous mi? True
transpose sonrası contiguous mi? False
view() hatasi: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
reshape() calisti, yeni shape: torch.Size([320, 64])
